# Credit Risk Scorecard — SQL Data Extraction Simulation

**Author:** Mathias Alejandro Gómez Chan  
**Date:** April 2026  
**GitHub:** [Mathias70473](https://github.com/Mathias70473)

---

> **Note:** This notebook is numbered `00` because it runs before any analysis. 
It replicates a production environment where data extraction precedes modeling, 
and feeds directly into `01-eda.ipynb`.

> **Outputs cleared on purpose.** Re-run top to bottom to regenerate results, then commit with fresh outputs.


In [ ]:
import sqlite3
import pandas as pd

# Load data
df = pd.read_csv('../data/raw/german_credit_data.csv')

# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')
df.to_sql('german_credit', conn, index=False, if_exists='replace')

print('Database created successfully')
print(f'Table shape: {df.shape}')

## Production Context

In a production credit risk environment, loan data does not arrive as a flat CSV. 
It lives in relational databases: joining tables for loan origination, customer 
demographics, credit bureau data, and transaction history.

This notebook simulates that extraction layer using SQLite. The German Credit Dataset 
is loaded into an in-memory SQL database, queried with analytical credit risk questions, 
and the validated output is passed to the EDA notebook.

**Target convention:** `kredit = 1` is a **good** customer and `kredit = 0` is a **default**. 
Every query defines default explicitly as `kredit = 0`.

**Key variable codes (from the data dictionary):**

- `laufkont` (checking status): 1 = no checking account · 2 = < 0 DM · 3 = 0–200 DM · 4 = ≥ 200 DM / salary 1yr+
- `moral` (credit history): 0 = delay in past · 1 = critical/other credits · 2 = none taken/all paid · 3 = existing paid duly · 4 = all paid at this bank
- `laufzeit` = duration (months) · `hoehe` = amount

## Query 1 — Portfolio Default Rate

**Business question:** What is the baseline default rate of the portfolio?

**Why it matters:** Before any modeling, a credit risk team needs the baseline default rate. 
At ~30% this portfolio is far riskier than a typical retail book (5–15%), which informs model 
calibration and approval thresholds. This query returns the actual rate, not just counts.

In [ ]:
# Query 1 — Portfolio default rate
query1 = """
SELECT
    COUNT(*)                                          AS total_customers,
    SUM(CASE WHEN kredit = 0 THEN 1 ELSE 0 END)       AS defaults,
    SUM(CASE WHEN kredit = 1 THEN 1 ELSE 0 END)       AS good,
    ROUND(AVG(CASE WHEN kredit = 0 THEN 1.0 ELSE 0 END), 3) AS default_rate
FROM german_credit
"""

result1 = pd.read_sql(query1, conn)
print('Query 1 — Portfolio default rate')
print(result1)

## Query 2 — Default Rate by Checking Account Status

**Business question:** How does the default rate vary across checking account status, 
and which segments are riskiest relative to the rest?

**Why it matters:** Checking account status is the strongest predictor in this dataset 
(IV = 0.666). This query computes the real default rate per category, labels each code, then uses 
window functions to **rank** segments and measure each one's **lift versus the average segment**. 
Note the known quirk of this dataset: *no checking account* (code 1) carries the highest default rate.

*Technique: CTE + `CASE WHEN` labels + `RANK() OVER (...)` + `AVG(...) OVER ()`.*

In [ ]:
# Query 2 — Default rate by checking status, labeled, ranked, with lift vs average segment
query2 = """
WITH segment_rates AS (
    SELECT
        laufkont AS checking_status,
        CASE laufkont
            WHEN 1 THEN 'No checking account'
            WHEN 2 THEN '< 0 DM'
            WHEN 3 THEN '0-200 DM'
            WHEN 4 THEN '>= 200 DM / salary 1yr+'
        END AS checking_label,
        COUNT(*) AS total,
        SUM(CASE WHEN kredit = 0 THEN 1 ELSE 0 END) AS defaults,
        AVG(CASE WHEN kredit = 0 THEN 1.0 ELSE 0 END) AS default_rate
    FROM german_credit
    GROUP BY laufkont
)
SELECT
    checking_status,
    checking_label,
    total,
    defaults,
    ROUND(default_rate, 3)                             AS default_rate,
    RANK() OVER (ORDER BY default_rate DESC)           AS risk_rank,
    ROUND(default_rate - AVG(default_rate) OVER (), 3) AS lift_vs_avg_segment
FROM segment_rates
ORDER BY risk_rank
"""

result2 = pd.read_sql(query2, conn)
print('Query 2 — Default rate by checking account status')
print(result2)

## Query 3 — Default Rate by Loan Duration Segment

**Business question:** Do longer loans carry higher default risk?

**Why it matters:** Loan duration is one of the most intuitive risk drivers in retail credit. 
This query uses `CASE WHEN` to bucket loans into duration segments and computes the default 
rate and average amount per segment.

*Technique: `CASE WHEN` segmentation + default rate per segment.*

In [ ]:
# Query 3 — Duration segments with default rate
query3 = """
SELECT
    CASE
        WHEN laufzeit <= 12 THEN '01: 0-12 months'
        WHEN laufzeit <= 24 THEN '02: 13-24 months'
        WHEN laufzeit <= 36 THEN '03: 25-36 months'
        ELSE                      '04: 37+ months'
    END AS duration_segment,
    COUNT(*)                                               AS total,
    ROUND(AVG(hoehe), 2)                                   AS avg_amount,
    SUM(CASE WHEN kredit = 0 THEN 1 ELSE 0 END)            AS defaults,
    ROUND(AVG(CASE WHEN kredit = 0 THEN 1.0 ELSE 0 END), 3) AS default_rate
FROM german_credit
GROUP BY duration_segment
ORDER BY duration_segment
"""

result3 = pd.read_sql(query3, conn)
print('Query 3 — Default rate by loan duration segment')
print(result3)

## Query 4 — Rule-Based High-Risk Screening (pre-scorecard)

**Business question:** Which *applicants* combine multiple high-risk characteristics and 
should be flagged for manual review **before** the scorecard is applied?

**Why it matters:** At application time the outcome is unknown, so a screening rule must use 
only application-time attributes — never the default flag. This rule flags applicants with all 
three risk traits at once:

- **No checking account or negative balance** — `laufkont IN (1, 2)`, the two highest-default segments (~49% and ~39%)
- **Long duration** — `laufzeit > 24` months
- **Weak credit history** — `moral IN (0, 1)`: *delay in past* or *critical account* (the two worst codes per the data dictionary)

No outcome (`kredit`) appears in the filter, so the rule is valid at application time.

In [ ]:
# Query 4 — Rule-based high-risk screening (no outcome leakage)
query4 = """
WITH flagged AS (
    SELECT
        laufkont AS checking_status,
        CASE laufkont WHEN 1 THEN 'No checking account' WHEN 2 THEN '< 0 DM' END AS checking_label,
        laufzeit AS duration_months,
        hoehe    AS amount,
        moral    AS credit_history,
        CASE moral WHEN 0 THEN 'Delay in past' WHEN 1 THEN 'Critical/other credits' END AS history_label
    FROM german_credit
    WHERE laufkont IN (1, 2)   -- no checking account or negative balance
      AND laufzeit > 24        -- long duration
      AND moral IN (0, 1)      -- weakest credit-history codes
)
SELECT *
FROM flagged
ORDER BY duration_months DESC, amount DESC
LIMIT 20
"""

result4 = pd.read_sql(query4, conn)
print(f'Query 4 — High-risk applicants flagged for manual review: {len(result4)} shown (top 20)')
print(result4)

## Pipeline Connection & Validation

Before handing off to `01-eda.ipynb`, the extraction layer runs basic data-quality checks 
(row count, nulls, duplicates). Only if the checks pass is the clean dataframe exported. 
This mirrors how a real pipeline gates bad data before it reaches the modeling stage.

In [ ]:
# Pipeline connection — validate, then export for notebook 01
df_clean = pd.read_sql('SELECT * FROM german_credit', conn)

n_rows  = df_clean.shape[0]
n_nulls = int(df_clean.isnull().sum().sum())
n_dups  = int(df_clean.duplicated().sum())

print(f'Rows: {n_rows} | Columns: {df_clean.shape[1]}')
print(f'Null values: {n_nulls} | Duplicate rows: {n_dups}')

if n_nulls == 0:
    df_clean.to_csv('../data/raw/german_credit_data.csv', index=False)
    print('Validation passed — data exported for 01-eda.ipynb')
else:
    print('Validation flagged nulls — review before export')

conn.close()